# `pv_lag` e il ritardo di reazione

Il forecaster non riceve covariate all'ora che deve prevedere, quindi all'arrivo
di un evento le ultime ore di input descrivono ancora la situazione precedente.
Sulla run completa questo si misura: all'ora di arrivo della nube il modello
riproduce solo circa due quinti del crollo, recupera in poche ore, e poi resta
ancorato al livello depresso mentre la produzione risale.

`pv_lag_pvgis` è la feature che porta esplicitamente la produzione delle ore
precedenti. L'ipotesi è che sia lei a spingere il modello a insistere sul
passato. Qui la stessa analisi viene applicata alle due run:

| run | feature set |
|---|---|
| `paper_faithful_gaussian_..._ep60_seed1` | `full` (con `pv_lag_pvgis`) |
| `paper_faithful_gaussian_no_pv_lag_..._ep60_seed1` | `no_pv_lag` |

Le due run vivono nella stessa `outputs/`, quindi il confronto non richiede di
cambiare branch: si leggono i rispettivi `predictions.csv`.

La metrica di confronto è la **quota di crollo catturata**, cioè quanta parte
del calo reale la previsione riproduce a quel lag. È adimensionale, quindi resta
confrontabile fra modelli con feature diverse; il bias in watt no.

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'physiq_pv').is_dir():
    raise FileNotFoundError('Avviare il notebook dalla root del repository o da notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import physiq_pv.reporting.event_onset as event_onset
import physiq_pv.reporting.report_dump as report_dump

event_onset = importlib.reload(event_onset)
report_dump = importlib.reload(report_dump)

RUNS = {
    'con pv_lag': 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_seed1',
    'senza pv_lag': 'pvgis_stgnn_paper_faithful_gaussian_no_pv_lag_detector_mtgflow_ep60_seed1',
}
run_dirs = {name: ROOT / 'outputs' / value for name, value in RUNS.items()}

for name, folder in run_dirs.items():
    predictions = folder / 'predictions.csv'
    print(f"{name:14s} {'OK     ' if predictions.is_file() else 'MANCANTE'} {predictions}")

## 1. Profilo di reazione, run per run

`ONSET_DAYS` sono i giorni dell'evento, `EXCLUDE_DAYS` quelli di salita che non
devono entrare nella baseline (se ci entrassero abbasserebbero il riferimento e
nasconderebbero il calo). L'arrivo è calcolato **per nodo**: la nube non tocca
tutte le località alla stessa ora.

In [ ]:
EVENT_NAME = '2019-04-24'
ONSET_DAYS = ['2019-04-24']
EXCLUDE_DAYS = ['2019-04-22', '2019-04-23']
BASELINE_DAYS = 12
SHORTFALL = 0.4
MAX_LAG = 10

responses = {}
for name, folder in run_dirs.items():
    if not (folder / 'predictions.csv').is_file():
        print(f'[skip] {name}: predictions.csv assente')
        continue
    window = event_onset.load_event_window(
        folder, event_days=ONSET_DAYS, baseline_days=BASELINE_DAYS,
        exclude_days=EXCLUDE_DAYS,
    )
    responses[name] = event_onset.build_onset_response(
        window, shortfall=SHORTFALL, max_lag_hours=MAX_LAG,
    )
    print(f"{name}: nodi raggiunti {responses[name]['n_nodes_reached']}, "
          f"mai raggiunti {responses[name]['unaffected']['n_nodes']}")

for name, response in responses.items():
    print(f'\n=== {name} ===')
    display(response['profile'][
        ['lag_hours', 'n', 'bias', 'mae', 'over_share', 'captured_share', 'picp']
    ].round(3))

## 2. Confronto

Se `pv_lag_pvgis` è la causa del ritardo, la run senza quella feature deve
catturare **più** crollo ai lag bassi. Se le due curve coincidono, il ritardo
viene dal resto della finestra di input, cioè dalle variabili meteo passate, e
togliere la feature non serve.

In [ ]:
comparison = event_onset.compare_onset_responses(responses)

for metric in ('captured_share', 'bias', 'mae', 'picp', 'over_share'):
    if metric not in comparison:
        continue
    print(f'\n=== {metric.upper()} ===')
    display(comparison.pivot(index='lag_hours', columns='run', values=metric).round(3))

In [ ]:
fig = event_onset.plot_onset_comparison(
    comparison, title=f'Reazione all evento {EVENT_NAME}: con e senza pv_lag'
)
figure_dir = run_dirs[next(iter(run_dirs))] / 'figures' / 'pv_lag_ablation'
figure_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_dir / f'onset_comparison_{EVENT_NAME}.png'
fig.savefig(figure_path, dpi=140, bbox_inches='tight')
plt.close(fig)
print(figure_path)
display(Image(filename=str(figure_path)))

In [ ]:
# Sintesi: quanto crollo cattura ciascuna run nelle prime ore, e quando rientra.
rows = []
for name, response in responses.items():
    profile = response['profile']
    early = profile.loc[profile['lag_hours'] <= 2, 'captured_share'].mean()
    peak_bias = profile.loc[profile['bias'].abs().idxmax()]
    settled = profile.loc[
        (profile['lag_hours'] > 0)
        & (profile['bias'].abs() < 0.25 * abs(peak_bias['bias']))
    ]
    rows.append({
        'run': name,
        'catturato_lag0': float(profile.loc[profile['lag_hours'] == 0, 'captured_share'].iloc[0]),
        'catturato_lag0_2': float(early),
        'bias_massimo_W': float(peak_bias['bias']),
        'lag_bias_massimo': int(peak_bias['lag_hours']),
        'lag_rientro': int(settled['lag_hours'].min()) if not settled.empty else None,
        'picp_min': float(profile['picp'].min()) if 'picp' in profile else None,
    })
summary = pd.DataFrame(rows)
display(summary.round(3))

## 3. Riepilogo da copiare

In [ ]:
report_dump.dump_sections({
    'confronto_profili': comparison,
    'sintesi_pv_lag': summary if 'summary' in dir() else None,
}, path=run_dirs[next(iter(run_dirs))] / 'analysis_summary_pv_lag.txt', max_rows=60)